In [ ]:
import pkgutil
import importlib.machinery

pkgutil.ImpImporter = importlib.machinery.BuiltinImporter

from vpython import *
import numpy as np

# ============================================
# ROLLER COASTER ENERGY ANALYZER
# Physics Simulation with VPython
# ============================================

# Simulation parameters
dt = 0.01  # time step
g = 9.8  # gravity (m/s^2)
mass = 100  # mass of cart (kg)
mu = 0.05  # coefficient of friction
track_length = 50  # length of track in x-direction

# Create the scene
scene = canvas(title='Roller Coaster Energy Analyzer',
               width=1200, height=600,
               center=vector(25, 15, 0),
               background=color.white,
               range=35)

# Create the track (a curved path)
def track_function(x):
    """Define the track shape - a roller coaster with hills"""
    return 15 + 8*np.sin(0.2*x) + 3*np.sin(0.5*x) - 0.1*x

# Create visual representation of track
track_objects = []

# Create the rails (two parallel lines)
for x in np.arange(0, track_length, 0.5):
    y = track_function(x)
    # Left rail
    sphere(pos=vector(x, y, -1), radius=0.15, color=color.gray(0.5))
    # Right rail
    sphere(pos=vector(x, y, 1), radius=0.15, color=color.gray(0.5))
    
    # Add cross ties every 2 meters
    if x % 2 < 0.25:
        box(pos=vector(x, y-0.2, 0), 
            size=vector(0.3, 0.1, 2.2), 
            color=color.gray(0.3))

# Create support pillars
for x in np.arange(2, track_length, 5):
    y = track_function(x)
    if y > 5:  # Only add pillars for elevated sections
        pillar = cylinder(pos=vector(x, 0, 0),
                          axis=vector(0, y-0.5, 0),
                          radius=0.3,
                          color=color.gray(0.7))
        # Add a base
        box(pos=vector(x, 0.2, 0), 
            size=vector(1, 0.4, 1), 
            color=color.gray(0.5))

# Add a ground plane
ground = box(pos=vector(track_length/2, -0.5, 0),
             size=vector(track_length+10, 0.5, 20),
             color=color.green)

# Create the cart - make it more visible
cart_body = box(pos=vector(0, track_function(0)+0.8, 0),
                size=vector(2.0, 1.0, 1.5),
                color=color.red)

# Add a roof
cart_roof = pyramid(pos=vector(0, track_function(0)+1.3, 0),
                    size=vector(1.8, 0.4, 1.3),
                    color=color.blue)

# Add wheels
wheel_radius = 0.3
wheel_positions = [
    (-0.7, -0.4, 0.8),  # front right
    (0.7, -0.4, 0.8),   # rear right
    (-0.7, -0.4, -0.8), # front left
    (0.7, -0.4, -0.8)   # rear left
]

wheels = []
for wx, wy, wz in wheel_positions:
    wheel = cylinder(pos=vector(wx, track_function(0)+wy, wz),
                     axis=vector(0, 0, 0.1),
                     radius=wheel_radius,
                     color=color.black)
    wheels.append(wheel)

# Group all cart parts
cart_parts = [cart_body, cart_roof] + wheels

# Create energy display in top-left corner
energy_display = label(pos=vector(-15, 25, 0),
                       text='',
                       xoffset=0, 
                       yoffset=0,
                       height=14, 
                       color=color.black,
                       line=False, 
                       box=False,
                       border=4,
                       font='monospace')

# Create a trail for the cart
trail = []
trail_length = 200

# Physics variables
x = 0.0  # starting position
v = 12.0  # initial velocity (m/s)
t = 0.0
total_work_friction = 0

print("Starting Roller Coaster Simulation...")
print(f"Initial Position: x=0, y={track_function(0):.1f}m")
print(f"Initial Velocity: {v} m/s")
print("Watch the cart move in the VPython window!")
print("Press Ctrl+C in the terminal to stop early")

try:
    while x < track_length - 2 and v > 0.1:  # Stop before the end or if cart almost stops
        rate(60)  # Control animation speed
        
        # Get current y position and slope
        y = track_function(x)
        
        # Calculate slope using numerical derivative
        dx_small = 0.01
        slope = (track_function(x + dx_small) - track_function(x - dx_small)) / (2 * dx_small)
        theta = np.arctan(slope)
        
        # Calculate rotation angle for wheels
        rotation_angle = (v * dt) / wheel_radius
        
        # Update cart position
        cart_body.pos = vector(x, y + 0.8, 0)
        cart_roof.pos = vector(x, y + 1.3, 0)
        
        # Update wheel positions and rotate them
        for i, wheel in enumerate(wheels):
            wx, wy, wz = wheel_positions[i]
            wheel.pos = vector(x + wx, y + 0.8 + wy, wz)
            wheel.rotate(angle=rotation_angle, axis=vector(0, 0, 1))
        
        # Rotate cart to match track slope
        cart_body.axis = vector(1, slope, 0)
        cart_body.up = vector(0, 1, 0)
        cart_roof.axis = vector(1, slope, 0)
        cart_roof.up = vector(0, 1, 0)
        
        # Add point to trail
        trail_sphere = sphere(pos=vector(x, y+0.8, 0), 
                              radius=0.15, 
                              color=color.yellow,
                              opacity=0.4)
        trail.append(trail_sphere)
        
        # Remove old trail points
        if len(trail) > trail_length:
            old_sphere = trail.pop(0)
            old_sphere.visible = False
        
        # ========================================
        # PHYSICS CALCULATIONS
        # ========================================
        
        # Forces
        normal = mass * g * abs(np.cos(theta))  # Normal force magnitude
        
        # Friction force
        friction = mu * normal if abs(v) > 0 else 0
        
        # Component of gravity along the track
        gravity_component = -mass * g * np.sin(theta)
        
        # Net force along the track
        net_force = gravity_component - (friction * np.sign(v))
        
        # Acceleration
        a = net_force / mass
        
        # Numerical integration (Euler's method)
        v = v + a * dt
        x = x + v * dt
        t = t + dt
        
        # ========================================
        # ENERGY CALCULATIONS
        # ========================================
        
        # Kinetic Energy
        ke = 0.5 * mass * v**2
        
        # Potential Energy
        pe = mass * g * y
        
        # Total Mechanical Energy
        te = ke + pe
        
        # Work done by friction
        work_friction = -abs(friction * v * dt)
        total_work_friction += work_friction
        
        # Update energy display
        energy_display.text = f"""╔══════════════════════════════════╗
║    ROLLER COASTER ENERGY      ║
╠══════════════════════════════════╣
║ KINETIC ENERGY: {ke:8.0f} J    ║
║ POTENTIAL ENERGY: {pe:8.0f} J    ║
║ TOTAL ENERGY: {te:8.0f} J    ║
║ WORK BY FRICTION: {total_work_friction:8.0f} J    ║
╠══════════════════════════════════╣
║ Time: {t:5.1f}s  Position: {x:5.1f}m ║
║ Velocity: {v:5.1f} m/s  Hill: {np.degrees(theta):5.1f}° ║
╚══════════════════════════════════╝"""

except KeyboardInterrupt:
    print("\nSimulation stopped by user")

# Simple final output
print("\n" + "="*40)
print("SIMULATION COMPLETE")
print("="*40)
print(f"Final Position: x={x:.1f}m")
print(f"Final Velocity: {v:.1f} m/s")
print(f"Total Time: {t:.1f} seconds")
print(f"Total Work by Friction: {total_work_friction:.0f} J")

print("\n✅ Roller coaster simulation finished!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>